# Module C, sequence 3: analysis (exercises)

Replace each `...` by what its comment asks, then run the cell. The steps follow the TP 3 of the handout of the module C. The corrected notebook is in the branch `correction-module-c`.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from pymodaq_utils.math_utils import gauss1D
from pymodaq_data.h5modules.data_saving import DataLoader

DATA = Path('../data')
SCAN = '/RawData/Scan000/Detector000/Data0D/CH00/Data00'  # one scan
SCAN8 = '/RawData/Scan001/Detector000/Data0D/CH00/Data00'  # the same scan repeated 8 times (scan_average = 8)


def model(x, amp, x0, fwhm, offset):
    return amp * gauss1D(x, x0, fwhm) + offset  # FWHM is the width parameter of gauss1D

## 1. Load the scan file

In [ ]:
loader = DataLoader(DATA / 'scan_photodiode.h5')
scan = ...  # the data of SCAN, with its navigation axes
scan8 = ...  # the data of SCAN8, with its navigation axes
loader.close_file()
print(scan, scan.axes)  # compare the shapes and the axes: the Average axis is only in the second one
print(scan8, scan8.axes)

## 2. Look at it and find the lines

In [ ]:
mean8 = ...  # the mean over the first navigation axis (the repetitions)
wavelength = ...  # the values of the wavelength axis of mean8
fig, ax = plt.subplots()
ax.plot(wavelength, scan.data[0], '.', label='one scan')
ax.plot(wavelength, mean8.data[0], '-', label='mean of 8 scans')
ax.set_xlabel('wavelength (nm)')
ax.legend()
peaks, _ = find_peaks(mean8.data[0], height=1.0)
print(wavelength[peaks])  # about 471, 529 and 590 nm

## 3. Crop around the main line and fit with scipy (uncertainties from the covariance)

In [ ]:
window = ...  # mean8 between 505 and 555 nm (by values, on the navigation axis)
...  # declare the navigation axis as a signal axis: `window` becomes a 1D data
x = window.get_axis_from_index(0)[0].get_data()
y = window.data[0]
p0 = [y.max() - y.min(), x[np.argmax(y)], 5., y.min()]
popt, pcov = ...  # fit `model` to (x, y) starting from p0
perr = ...  # the uncertainties of the parameters, from pcov
for name, value, error in zip(['amplitude', 'center (nm)', 'FWHM (nm)', 'offset'], popt, perr):
    print(f'{name}: {value:.3f} +/- {error:.3f}')

## 4. The same fit with DataWithAxes.fit

In [ ]:
fit = ...  # fit `window` with `model`, starting from p0, with the method of the data object
print(fit.fit_coeffs[0])
assert np.allclose(fit.fit_coeffs[0], popt, rtol=1e-3)  # what is missing compared to curve_fit?

## 5. Fit the whole scan, without cropping

Compare the background (offset) with the one of the cropped fit: what do the satellite lines do? On the whole scan
you will also meet a point that the cropped window did not contain.

In [ ]:
full = ...  # a copy of mean8...
...  # ...whose navigation axis is declared as a signal axis
x_full = full.get_axis_from_index(0)[0].get_data()
y_full = full.data[0]
finite = ...  # mask of the finite values: curve_fit refuses NaN (look at np.isnan(scan8.data[0]): what happened?)
popt_full, pcov_full = curve_fit(model, x_full[finite], y_full[finite], p0=p0)
print('offset, cropped:', popt[3], ' whole scan:', popt_full[3])

## 6. Uncertainties from the repetitions, weighted fit

In [ ]:
n_repeat = scan8.shape[0]
mean8.errors = ...  # the standard error of the mean of the 8 repetitions (std with ddof=1, divided by sqrt(8))
window = ...  # crop again between 505 and 555 nm: the errors follow the slicing
window.nav_indexes = ()
popt_w, pcov_w = ...  # weighted fit: sigma from the errors of `window`, absolute_sigma=True
chi2_red = np.sum(((window.data[0] - model(x, *popt_w)) / window.errors[0]) ** 2) / (len(x) - len(popt_w))
print(popt_w, np.sqrt(np.diag(pcov_w)))
print('reduced chi2:', chi2_red)  # larger than 1: why?

## 7. The axis in eV

In [ ]:
axis_ev = ...  # the axis of `window` converted to eV, without modifying it (context 'spectroscopy')
window_ev = window.deepcopy()
window_ev.axes[0] = axis_ev
x_ev = axis_ev.get_data()
y_ev = window_ev.data[0]
p0_ev = [10., x_ev[np.argmax(y_ev)], 0.02, y_ev.min()]
popt_ev, pcov_ev = curve_fit(model, x_ev, y_ev, p0=p0_ev)
print(axis_ev.units, popt_ev[1], popt_ev[2], np.sqrt(np.diag(pcov_ev))[1:3])  # centre and FWHM in eV

## 8. One spectrum with Naverage = 200: the errors come with the data (PyMoDAQ 5.3.9 or later)

In [ ]:
loader = DataLoader(DATA / 'spectrum_naverage200.h5')
spectrum = ...  # load everything under '/RawData/Detector000' and keep the data named 'Spectrum'
loader.close_file()
print(spectrum.errors[0][:3])

## 9. Compare with the standard deviation of the continuous recording

In [ ]:
loader = DataLoader(DATA / 'continuous_spectra.h5')
series = ...  # the enlargeable array '/RawData/Detector000/Data1D/CH00/EnlData00', with its navigation axis
loader.close_file()
n_spectra = series.shape[0]
sem_series = ...  # the standard deviation of the series over the acquisitions, divided by sqrt(n_spectra)
i_peak = int(np.argmax(spectrum.data[0]))
print('standard error at the peak, Naverage=200:', spectrum.errors[0][i_peak], ' series:', sem_series[i_peak])
print('standard error far from the peak, Naverage=200:', spectrum.errors[0][:10].mean(), ' series:', sem_series[:10].mean())
fig, ax = plt.subplots()
ax.plot(series.data[0][:, i_peak], '.-')  # find the slow drift and estimate its period
ax.set_xlabel('acquisition number')
ax.set_ylabel('intensity at the peak')

## Control question 3

At the top of the line, the standard error given by the DAQ_Viewer (200 acquisitions) and the standard deviation of
the series divided by √200 are close. Far from the line they are close too, but much smaller: why? What does the
comparison of the standard deviation at the top with the one far from the line say about the origin of the noise?

## 10. Going further (open, for those who are done)

No hole to fill here and no single answer: pick one or several items, write your own cells, and keep what you find
in your handout. Everything you need is in the files of the `data` folder and in the objects above.

1. **Model the drift.** In the continuous recording, fit the intensity at the peak versus the acquisition number with
   a sinusoid. What are its period (in acquisitions) and its relative depth? Divide the series by the fitted drift
   and compute again the standard deviation at the peak: what is left?
2. **All the lines at once.** Fit the whole averaged scan with three gaussians and one background. Compare the
   centre, the width and the background of the main line with the cropped fit of the part 3.
3. **The fit as a data object.** Build a `DataCalculated` holding the data of `window` and the fitted curve as two
   channels (same axis, labels 'data' and 'fit'), plot it with the PyMoDAQ viewer (`.plot('qt')`), then save it
   with its errors in an HDF5 file and open it in H5Browser.
4. **Navigation and signal on the series.** `series` has a navigation axis (time) and a signal axis (wavelength).
   Keep 525 to 532 nm on the signal axis with `vsig`: why does PyMoDAQ refuse (look at `series.distribution`)? Do it
   with `isig` instead (the axis has a `find_index` method), sum over the signal axis to get one number per
   acquisition, and compare with the intensity at the peak of the part 9. Then keep the first 50 acquisitions only.
5. **Your own measurement.** Record a scan of the photodiode yourself in the DAQ_Scan (module A), with the satellite
   lines and the drift switched on in the settings of the plugin, and run this notebook on your file.